# Lab 50: Simulacion de Cadena de Rydberg

Los **arrays de tweezer** con atomos de Rydberg son el paradigma cuantico con mayor numero de qubits actuales (QuEra: 10 000 atomos, 2024). Su fisica se basa en la interaccion $U(r) = C_6/r^6$ entre estados excitados de alto numero cuantico principal $n$.

En este lab simularemos:
1. El **Hamiltoniano PXP** de cadena de Rydberg con bloqueo fuerte.
2. Evolucion temporal y deteccion de la fase Z2 ordenada.
3. Diagrama de fase $\Omega$-$\Delta$.
4. Cicatrices cuanticas (quantum many-body scars).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
import warnings
warnings.filterwarnings('ignore')

np.random.seed(0)

I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Pg = np.array([[1,0],[0,0]], dtype=complex)  # |g><g|
Pr = np.array([[0,0],[0,1]], dtype=complex)  # |r><r|

def op_on_site(op, site, N):
    ops = [I2]*N; ops[site] = op
    result = ops[0]
    for o in ops[1:]: result = np.kron(result, o)
    return result

print("Hamiltoniano de Rydberg — cadena 1D")
print(f"Espacio de Hilbert para N=8: {2**8} estados")

## 1. Hamiltoniano PXP y Espectro

El Hamiltoniano completo de Rydberg incluye drive ($\Omega$), desintonizacion ($\Delta$) e interaccion ($C_6/r^6$):
$$H = \frac{\Omega}{2}\sum_i\sigma_i^x - \Delta\sum_i n_i + \sum_{i<j}\frac{C_6}{r_{ij}^6}n_in_j$$
En el limite de bloqueo fuerte el modelo se reduce al PXP:
$$H_\text{PXP} = \frac{\Omega}{2}\sum_i P_{i-1}X_iP_{i+1}$$

In [ ]:
def build_rydberg_H(N, Omega, Delta, C6=100.0, a=1.0):
    dim = 2**N
    H = np.zeros((dim, dim), dtype=complex)
    for i in range(N):
        H += (Omega/2) * op_on_site(X, i, N)
        H -= Delta * op_on_site(Pr, i, N)
    for i in range(N-1):
        ni = op_on_site(Pr, i, N)
        nj = op_on_site(Pr, i+1, N)
        H += C6 * (ni @ nj)  # solo 1er vecino (r=a, C6/a^6=C6 en unidades a=1)
    return H

def build_pxp_H(N, Omega=1.0):
    dim = 2**N
    H = np.zeros((dim, dim), dtype=complex)
    for i in range(N):
        il, ir = max(i-1, 0), min(i+1, N-1)
        PXP_i = op_on_site(Pg, il, N) @ op_on_site(X, i, N) @ op_on_site(Pg, ir, N)
        H += (Omega/2) * PXP_i
    return H

N4 = 4
H4 = build_pxp_H(N4)
evals4 = np.linalg.eigvalsh(H4)
print(f"PXP N=4: espectro = {evals4.round(4)}")
print(f"Hermítico: {np.allclose(H4, H4.conj().T)}")

# Espectro PXP para N=8
N8 = 8
H8 = build_pxp_H(N8)
evals8 = np.linalg.eigvalsh(H8)
fig, ax = plt.subplots(figsize=(8, 3))
ax.scatter(evals8, np.zeros_like(evals8), c='steelblue', s=30, alpha=0.6)
ax.set_xlabel('Energia E/Omega'); ax.set_title(f'Espectro PXP N={N8} — notar estados igualmente espaciados (cicatrices)')
ax.set_yticks([]); plt.tight_layout(); plt.show()
print(f"PXP N=8: {len(evals8)} autoestados, E_min={evals8[0]:.4f}, E_max={evals8[-1]:.4f}")

## 2. Diagrama de Fase $\Omega$-$\Delta$

Barremos $(\Omega, \Delta)$ y medimos en el estado fundamental el numero de Rydberg $\langle n\rangle$ y el orden Z2. Las fases son:
- **Paramagnet** ($\Delta < 0$): $\langle n\rangle \approx 0$
- **Z2 ordenado** ($\Delta/\Omega \approx 1.5$): patron alternante $|grgrg\ldots\rangle$
- **Saturacion** ($\Delta \gg \Omega$): $\langle n\rangle \approx 1$

In [ ]:
N_ph = 6
N_ryd_op = sum(op_on_site(Pr, i, N_ph) for i in range(N_ph)) / N_ph
Z2_op    = sum((-1)**i * op_on_site(Pr, i, N_ph) for i in range(N_ph)) / N_ph

Omega_vals = np.linspace(0.5, 3.0, 18)
Delta_vals = np.linspace(-2.0, 6.0, 22)

n_map = np.zeros((len(Delta_vals), len(Omega_vals)))
z2_map = np.zeros((len(Delta_vals), len(Omega_vals)))

print("Calculando diagrama de fase (puede tardar ~30s)...")
for j, Omega in enumerate(Omega_vals):
    for i, Delta in enumerate(Delta_vals):
        H = build_rydberg_H(N_ph, Omega, Delta, C6=100.0)
        evals_p, evecs_p = eigh(H, subset_by_index=[0,0])
        gs = evecs_p[:,0]
        n_map[i,j]  = float(np.real(gs.conj() @ N_ryd_op @ gs))
        z2_map[i,j] = float(np.abs(gs.conj() @ Z2_op @ gs))
print("Hecho.")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].pcolormesh(Omega_vals, Delta_vals, n_map, cmap='RdYlBu_r', shading='auto')
plt.colorbar(im0, ax=axes[0], label='<n_Ryd>/N')
axes[0].set_xlabel('Omega'); axes[0].set_ylabel('Delta')
axes[0].set_title(f'Densidad Rydberg (N={N_ph})')

im1 = axes[1].pcolormesh(Omega_vals, Delta_vals, z2_map, cmap='viridis', shading='auto')
plt.colorbar(im1, ax=axes[1], label='Orden Z2')
axes[1].set_xlabel('Omega'); axes[1].set_ylabel('Delta')
axes[1].set_title('Orden Z2 (fase alternante)')

# Linea Delta/Omega = 1.5
x_line = Omega_vals
axes[1].plot(x_line, 1.5*x_line, 'r--', lw=1.5, label='Delta/Omega=1.5')
axes[1].legend(fontsize=9)
plt.suptitle(f'Diagrama de Fase Rydberg (N={N_ph})', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

idx_max = np.unravel_index(z2_map.argmax(), z2_map.shape)
print(f"Maximo orden Z2 en Omega={Omega_vals[idx_max[1]]:.2f}, Delta={Delta_vals[idx_max[0]]:.2f}")

## 3. Evolucion Temporal y Cicatrices Cuanticas

Partimos del estado Z2 $|\mathbb{Z}_2\rangle = |grgrg\ldots\rangle$ y evolucionamos con $H_\text{PXP}$. Las **cicatrices cuanticas** producen **revivals** periodicos de alta fidelidad — fenomeno que viola la hipotesis ETH de termalización.

Los autoestados responsables son los de menor entropia de entrelazamiento en el espectro medio.

In [ ]:
N_ev = 8
H_pxp = build_pxp_H(N_ev)
evals_pxp, evecs_pxp = eigh(H_pxp)

# Estado Z2: |grgrgrg...> = bits alternos 0,1,0,1,...
z2_bits = [i % 2 for i in range(N_ev)]
z2_idx  = sum(b*(2**(N_ev-1-i)) for i,b in enumerate(z2_bits))
psi_z2  = np.zeros(2**N_ev, dtype=complex); psi_z2[z2_idx] = 1.0
c_z2 = evecs_pxp.T @ psi_z2

# Estado de vacio: |0000...>
psi0 = np.zeros(2**N_ev, dtype=complex); psi0[0] = 1.0
c_0  = evecs_pxp.T @ psi0

t_vals = np.linspace(0, 30, 400)
fid_z2 = []; fid_0  = []
ent_z2 = []

n_total_op = sum(op_on_site(Pr, i, N_ev) for i in range(N_ev)) / N_ev

print("Evolucionando...")
for t in t_vals:
    ph = np.exp(-1j * evals_pxp * t)
    psi_t_z2 = evecs_pxp @ (c_z2 * ph)
    psi_t_0  = evecs_pxp @ (c_0  * ph)
    fid_z2.append(abs(psi_t_z2.conj() @ psi_z2)**2)
    fid_0.append(float(np.real(psi_t_0.conj() @ n_total_op @ psi_t_0)))
    # Entanglement desde Z2
    pm = psi_t_z2.reshape(2**(N_ev//2), 2**(N_ev//2))
    sv = np.linalg.svd(pm, compute_uv=False); sv2 = sv**2; sv2 = sv2[sv2>1e-15]
    ent_z2.append(-np.sum(sv2*np.log2(sv2)))

# Entropias de todos autoestados
ents_auto = []
for k in range(min(len(evals_pxp), 150)):
    pm = evecs_pxp[:,k].reshape(2**(N_ev//2), 2**(N_ev//2))
    sv = np.linalg.svd(pm, compute_uv=False); sv2=sv**2; sv2=sv2[sv2>1e-15]
    ents_auto.append(-np.sum(sv2*np.log2(sv2)))

page_val = N_ev/2 - 1/(2*np.log(2))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].plot(t_vals, fid_z2, 'darkorange', lw=1.5)
axes[0,0].set_title('Revival |<Z2|psi(t)>|^2'); axes[0,0].set_xlabel('t'); axes[0,0].set_ylabel('Fidelidad')

axes[0,1].plot(t_vals, ent_z2, 'steelblue', lw=1.5)
axes[0,1].axhline(page_val, color='red', ls='--', label=f'Page={page_val:.2f}')
axes[0,1].set_title('Entropia de entrelazamiento desde Z2')
axes[0,1].set_xlabel('t'); axes[0,1].set_ylabel('S(L/2)'); axes[0,1].legend()

axes[1,0].plot(t_vals, fid_0, 'mediumseagreen', lw=1.5)
axes[1,0].set_title('n_Ryd desde estado vacio'); axes[1,0].set_xlabel('t'); axes[1,0].set_ylabel('<n>')

E_autos = evals_pxp[:len(ents_auto)]
ovlp_sq  = np.abs(c_z2[:len(ents_auto)])**2
scars = (ovlp_sq > 0.03) & (np.array(ents_auto) < page_val*0.7)
axes[1,1].scatter(E_autos, ents_auto, c='steelblue', s=15, alpha=0.5, label='Todos')
axes[1,1].scatter(E_autos[scars], np.array(ents_auto)[scars], c='red', s=60, zorder=3, label='Cicatrices')
axes[1,1].axhline(page_val, color='orange', ls='--', label=f'Page={page_val:.2f}')
axes[1,1].set_xlabel('Energia'); axes[1,1].set_ylabel('S(L/2)')
axes[1,1].set_title('Espectro de entropias (cicatrices en rojo)'); axes[1,1].legend(fontsize=8)

plt.suptitle(f'Dinamica y Cicatrices Cuanticas — PXP N={N_ev}', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Max revival Z2: {max(fid_z2):.4f}")
print(f"Cicatrices identificadas: {scars.sum()}")
print(f"Entropia Page (ETH): {page_val:.4f}")

## 4. Bloqueo de Rydberg y Radio Critico

El **radio de bloqueo** determina el alcance de la puerta CZ:
$$r_b = \left(\frac{C_6}{\hbar\Omega}\right)^{1/6}$$
Cuando dos atomos estan dentro de $r_b$, el estado doblemente excitado $|rr\rangle$ queda desplazado fuera de resonancia.

In [ ]:
# Radio de bloqueo y potencial de interaccion Rydberg
C6_GHz_um6 = 862000  # para Rb n=70

r_vals = np.linspace(2, 15, 200)  # um
U_vals = C6_GHz_um6 / r_vals**6   # GHz

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
Omega_ref = 1.0  # MHz = 0.001 GHz
rb = (C6_GHz_um6 / (Omega_ref * 1e-3))**(1/6)

axes[0].semilogy(r_vals, U_vals*1e3, 'steelblue', lw=2, label='U(r) = C6/r^6')
axes[0].axhline(Omega_ref, color='red', ls='--', lw=1.5, label=f'Omega = {Omega_ref} MHz')
axes[0].axvline(rb, color='green', ls='--', lw=1.5, label=f'r_b = {rb:.2f} um')
axes[0].set_xlabel('Distancia r (um)'); axes[0].set_ylabel('U(r) (MHz)')
axes[0].set_title('Potencial de interaccion Rydberg (Rb, n=70)')
axes[0].legend(fontsize=9); axes[0].set_xlim([2,15])

# Radio de bloqueo vs Omega
Omega_range = np.linspace(0.2, 5.0, 100)
rb_range = (C6_GHz_um6 / (Omega_range * 1e-3))**(1/6)
axes[1].plot(Omega_range, rb_range, 'darkorange', lw=2)
axes[1].axhline(5.0, color='gray', ls=':', label='Separacion tipica 5 um')
axes[1].set_xlabel('Omega (MHz)'); axes[1].set_ylabel('Radio de bloqueo r_b (um)')
axes[1].set_title('Radio de bloqueo vs frecuencia Rabi'); axes[1].legend()
plt.tight_layout(); plt.show()

print("Radio de bloqueo para Rb n=70:")
for Om in [0.5, 1.0, 2.0, 4.0]:
    rb_om = (C6_GHz_um6 / (Om * 1e-3))**(1/6)
    print(f"  Omega={Om} MHz -> r_b = {rb_om:.2f} um")

# Puerta CZ: tabla de verdad
print("
Puerta CZ (bloqueo de Rydberg) — tabla de verdad:")
CZ = np.diag([1.0, -1.0, 1.0, 1.0])
basis = ['|00>', '|01>', '|10>', '|11>']
for i, label in enumerate(basis):
    phase = CZ[i,i]
    print(f"  {label} -> {'+' if phase>0 else '-'}{label}")
print(f"Unitaria: {np.allclose(CZ@CZ, np.eye(4))}")

## Resumen

| Concepto | Resultado clave |
|---|---|
| Hamiltoniano Rydberg | Drive + desintonizacion + interaccion $C_6/r^6$ |
| Modelo PXP | Bloqueo fuerte: $H = (\Omega/2)\sum P_{i-1}X_iP_{i+1}$ |
| Fase Z2 | Orden alternante para $\Delta/\Omega \approx 1.5$; detectado por $|\langle S(\pi)\rangle|$ |
| Cicatrices cuanticas | Autoestados con $S < S_\text{Page}$; producen revivals de $|\mathbb{Z}_2\rangle$ |
| Radio de bloqueo | $r_b = (C_6/\hbar\Omega)^{1/6}$; Rb $n=70$, $\Omega=1$ MHz: $r_b \approx 8.4$ um |
| Puerta CZ | Diagonal: $\text{diag}(1,-1,1,1)$; fidelidad experimental $\geq 99.5\%$ |